# Inference & Optimization Pipeline II: Adam + L-BFGS on a *Self-Consistent* Galaxy

This notebook fits potential parameters (`pot_params`) and distribution-function parameters (`disk_df_params`, `bulge_df_params`) so that the Phoenix surrogate reproduces an observed galaxy, with the **Poisson self-consistency term switched on**.

**What is different from `optimization_pipeline.ipynb`:** there, the "observed" galaxy was built by picking the potential parameters and the DF parameters *independently*. That does **not** produce a dynamically self-consistent galaxy — the density of the DF-sampled tracers does not reproduce the density that sources the potential. Measuring the Poisson residual at those "true" parameters gave a large value, which means the truth was **not a minimum of the objective**: switching the physics term on actively pulled the fit *away* from the true parameters, and the optimizer could reach a *more* self-consistent state than the true galaxy itself.

So here the order of operations is fixed:

1. **Solve for a self-consistent ground truth** — adjust the baryonic potential until the DF's own density sources it (`make_self_consistent_truth`).
2. **Generate the mock observation from that self-consistent model.**
3. **Fit from a far-off start with the physics term on** (`w_poisson > 0`), in two
   optimizer stages: **Adam** (first order, with bandwidth annealing) to cross the
   plateau and get into the basin, then **L-BFGS** (quasi-Newton, with a Wolfe line
   search) at fixed bandwidth to polish the point Adam leaves behind.

Three further corrections are used throughout, each of which was measured at the true parameters:

| ingredient | why |
|---|---|
| `obs_bandwidth=` | During bandwidth annealing the **observation** is blurred to the model's current resolution. Previously only the model was smoothed, so the wide-bandwidth stages minimized a biased objective (the mass term was 0 at the truth at the data's bandwidth, but ~9.8 at h=8). |
| `frozen_params=` | `Sigma0` and `N0_spheroid` cancel *exactly* in the mass-normalized tracer weights (`w/sum(w)*M_disk`), and `L0`/`Rinit_for_Rc` do not affect these maps. All four have numerically zero gradient, so fitting them only lets them drift and inflates any recovery metric. |
| shared `poisson_kwargs` | Self-consistency is only defined relative to one penalty definition (grid, bandwidth, masking, kernel matching). Tuning the truth against one definition and fitting against another reintroduces the bias. |

The self-consistency penalty itself was also repaired: it now compares against the **baryonic** potential only (the tracers carry `M_disk + M_bulge` and cannot reproduce the dark halo), masks to pixels where tracers actually exist (empty pixels previously produced ~7-dex artefacts over two thirds of the grid), uses relative floors and the physical `G`, and can convolve the analytic density with the same kernel as the tracer KDE (`match_kernel=True`).
## What is different from `optimization_pipeline2.ipynb`

Only the optimizer. Section 6 is unchanged — the same annealed Adam run — and Section 7 adds a second stage on top of it.

The division of labour is the point. Adam is the right tool for the first stage: it is robust to the noisy, badly scaled gradients this loss produces far from the solution, and bandwidth annealing changes the objective at every step, which no curvature estimate can track. It is the wrong tool for the last stage: once the fit is inside the basin the loss surface is a set of long narrow valleys (the potential parameters' gradients are ~1000x the DF shape parameters'), and Adam's per-coordinate rescaling still takes a fixed-size step along a direction that ignores curvature, so it crawls along the valley floor. L-BFGS estimates the inverse Hessian from the last few (step, gradient-change) pairs and gets both the direction and — from the line search — the distance, which is exactly what a narrow valley needs.

The polish is applied to **both** Adam endpoints — the plateaued no-annealing fit of Section 5 and the annealed fit of Section 6 — which turns Section 5's claim into a testable one: if the plateau were merely a conditioning problem, a quasi-Newton method should be able to descend it. Section 7 says why it should not.

The one hard requirement is that **L-BFGS runs at a fixed bandwidth**: its curvature memory is only meaningful for a single fixed objective. It therefore picks up at `OBS_BANDWIDTH`, the bandwidth the annealed Adam run holds at for its last third, so both stages optimize the identical (and exact — the required blur is zero there) objective.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import optax
from jax.flatten_util import ravel_pytree
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from phoenix.actions_to_phasespace.actions_to_phasespace_nn import PhoenixMapper
from phoenix.optimization.observables import generate_edge_on_maps
from phoenix.optimization.pipeline import (
    make_observation,
    make_self_consistent_truth,
    fit,
    make_loss_fn,
    params_to_log,
    log_to_params,
    log_bounds_tree,
)

## 1. Setup

`POISSON_KWARGS` is defined **once** and used for both the self-consistency solve and the fit. `OBS_BANDWIDTH` is the bandwidth `make_observation` bins with — the annealing must never go below it, since the data carries no information on finer scales.

In [ ]:
pot_params_nominal = {
    'M_halo': 1e12, 'a_halo': 20.0,
    'M_disk': 5e10, 'a_disk': 3.0, 'b_disk': 0.3,
    'M_bulge': 1e10, 'a_bulge': 1.0,
}
disk_df_params_nominal = {
    "R0": 8.0, "Rd": 3.0, "Sigma0": 1000.0,
    "RsigR": 6.0, "RsigZ": 6.0,
    "sigmaR0_R0": 35.0, "sigmaz0_R0": 20.0,
    "L0": 10.0, "Rinit_for_Rc": 8.0,
}
bulge_df_params_nominal = {
    "N0_spheroid": 1e10, "J0_spheroid": 100.0,
    "Gamma_spheroid": 1.5, "Beta_spheroid": 4.5, "eta_spheroid": 1.0,
}

mapper = PhoenixMapper()

SEED = 0
N_PARTICLES = 6_000
GRID_SIZE = 14
EXTENT_X, EXTENT_Z = 15.0, 10.0

# Bandwidth the observation is binned at; the anneal stops here.
OBS_BANDWIDTH = 0.25 * max(2 * EXTENT_X / GRID_SIZE, 2 * EXTENT_Z / GRID_SIZE)

# ONE shared definition of the self-consistency penalty.
POISSON_KWARGS = dict(grid_size=20, match_kernel=True, n_quad=3)

W_POISSON = 0.1                                              # physics term ON
FROZEN = ('Sigma0', 'N0_spheroid', 'Rinit_for_Rc', 'L0')     # zero-gradient params

GRID_KW = dict(N_disk=N_PARTICLES, N_bulge=N_PARTICLES, grid_size=GRID_SIZE,
               extent_x=EXTENT_X, extent_z=EXTENT_Z, prng_seed=SEED)

print(f"observation bandwidth / anneal floor: {OBS_BANDWIDTH:.3f} kpc")

In [ ]:
def plot_maps_row(axes, maps, mask, title_prefix, vmax_mass):
    extent = [-EXTENT_X, EXTENT_X, -EXTENT_Z, EXTENT_Z]
    mass = np.where(mask, np.array(maps['mass']), np.nan)
    vrot = np.where(mask, np.array(maps['v_rot']), np.nan)
    sigma = np.where(mask, np.array(maps['sigma']), np.nan)
    im0 = axes[0].imshow(mass, origin='lower', extent=extent, cmap='magma',
                         norm=LogNorm(vmin=max(vmax_mass * 1e-4, 1e3), vmax=vmax_mass),
                         aspect='auto')
    axes[0].set_title(f'{title_prefix}: mass')
    im1 = axes[1].imshow(vrot, origin='lower', extent=extent, cmap='seismic',
                         vmin=-220, vmax=220, aspect='auto')
    axes[1].set_title(f'{title_prefix}: v_rot')
    im2 = axes[2].imshow(sigma, origin='lower', extent=extent, cmap='viridis',
                         vmin=0, vmax=150, aspect='auto')
    axes[2].set_title(f'{title_prefix}: sigma')
    return im0, im1, im2


def recovery(res, truth):
    """Relative errors, reported separately for the potential parameters, the DF
    parameters, and the structurally unconstrained (frozen) ones -- lumping them into
    one median hides which part of the model is actually being recovered."""
    fitted = {**res['pot_params'], **res['disk_df_params'], **res['bulge_df_params']}
    errs = {k: abs(float(fitted[k]) - truth[k]) / abs(truth[k]) * 100 for k in truth}
    pot_errs = {k: errs[k] for k in res['pot_params']}
    con_errs = {k: v for k, v in errs.items() if k not in FROZEN}
    return errs, pot_errs, con_errs


def report(tag, res, truth):
    errs, pot_errs, con = recovery(res, truth)
    L = np.array(res['history']['loss'])
    print(f"{tag}")
    print(f"    loss {L[0]:8.3f} -> {L[-1]:8.4f}"
          + ("   [NaN present!]" if np.isnan(L).any() else ""))
    print(f"    poisson residual at end: {res['history']['poisson_penalty'][-1]:.4f}")
    print(f"    POTENTIAL (7):  median {np.median(list(pot_errs.values())):6.1f}%   "
          f"within 20%: {sum(v < 20 for v in pot_errs.values())}/7")
    print(f"    all constrained (17): median {np.median(list(con.values())):6.1f}%   "
          f"within 20%: {sum(v < 20 for v in con.values())}/17")
    return errs

## 2. Making the Ground Truth Self-Consistent

`make_self_consistent_truth` adjusts the **baryonic** potential (`M_disk`, `a_disk`, `b_disk`, `M_bulge`, `a_bulge`) until the density of the DF-sampled tracers reproduces the density that sources it. The halo parameters are deliberately left alone: the halo is dark, the tracers carry only `M_disk + M_bulge`, and asking them to reproduce a halo that outweighs them by more than an order of magnitude can never be satisfied.

Watch the residual drop — that gap is exactly what used to bias the fit.

In [ ]:
sc = make_self_consistent_truth(
    mapper, pot_params_nominal, disk_df_params_nominal, bulge_df_params_nominal,
    tune_params=('M_disk', 'a_disk', 'b_disk', 'M_bulge', 'a_bulge'),
    N_disk=N_PARTICLES, N_bulge=N_PARTICLES, prng_seed=SEED,
    learning_rate=0.02, n_steps=250,
    poisson_kwargs=POISSON_KWARGS,
)

pot_params_true = sc['pot_params']
disk_df_params_true = sc['disk_df_params']
bulge_df_params_true = sc['bulge_df_params']
truth = {**pot_params_true, **disk_df_params_true, **bulge_df_params_true}

print(f"Poisson residual at the ground truth: {sc['penalty_initial']:.4f} -> {sc['penalty_final']:.4f}")
print(f"{'parameter':10s} {'nominal':>12s} {'self-consistent':>16s}")
for k in ('M_disk', 'a_disk', 'b_disk', 'M_bulge', 'a_bulge'):
    print(f"{k:10s} {pot_params_nominal[k]:12.4g} {pot_params_true[k]:16.4g}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(sc['history'], color='tab:purple')
ax.set_yscale('log'); ax.set_xlabel('iteration'); ax.set_ylabel('Poisson residual')
ax.set_title('Solving for a self-consistent ground truth')
plt.tight_layout(); plt.show()

### A caveat worth reading

The self-consistent solution is a genuinely *different* galaxy from the nominal one: this DF, sampled in this potential family, implies a considerably thicker and more extended disk. Watch whether `a_bulge` ends up pinned at its lower bound (`DEFAULT_PARAM_BOUNDS` allows 0.05 kpc) — if it does, the self-consistent bulge is implausibly compact and the bound is doing the work rather than the physics.

If you would rather keep a thin-disk truth, tune the **DF** parameters against a fixed potential instead by passing e.g. `tune_params=('Rd', 'RsigR', 'RsigZ', 'sigmaR0_R0', 'sigmaz0_R0')`. Either direction gives a self-consistent pair; they just differ in which side you treat as given.

## 3. The Observed Galaxy

The mock observation is generated from the **self-consistent** parameters. This is the one change that makes the physics term legitimate: the truth is now (approximately) a minimum of the self-consistency penalty, so the term no longer fights the data.

In [ ]:
obs_maps = make_observation(mapper, pot_params_true, disk_df_params_true,
                            bulge_df_params_true, **GRID_KW)

mass_mask = np.array(obs_maps['mass']) > 1e-3
vmax_mass = float(np.nanmax(np.array(obs_maps['mass'])))
print(f"{int(mass_mask.sum())}/{GRID_SIZE**2} pixels above the detection floor")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ims = plot_maps_row(axes, obs_maps, mass_mask, 'Observed', vmax_mass)
for im, ax, lab in zip(ims, axes, ['Density', 'km/s', 'km/s']):
    fig.colorbar(im, ax=ax, label=lab)
plt.tight_layout(); plt.show()

## 4. A Far-Off Starting Guess

The same hard regime as before: potential parameters ~5.3x too large, the disk DF 2.5x too large, the bulge DF 10x too small.

In [ ]:
pot_params_init = {k: v * 5.3 for k, v in pot_params_true.items()}
disk_df_params_init = {k: v * 2.5 for k, v in disk_df_params_true.items()}
bulge_df_params_init = {k: v * 0.1 for k, v in bulge_df_params_true.items()}

init_maps = generate_edge_on_maps(mapper, pot_params_init, disk_df_params_init,
                                 bulge_df_params_init, **GRID_KW)

FIT_KW = dict(**GRID_KW, poisson_kwargs=POISSON_KWARGS,
              loss_weights=(1.0, 1.0, 1.0, W_POISSON),
              learning_rate=0.05, frozen_params=FROZEN)

## 5. Without Annealing: the Plateau

First, a fit at a **fixed** (sharp) bandwidth. The model and observed maps barely overlap from this far start, so the masked log-mass term has a vanishing gradient: a high, flat plateau the optimizer cannot descend.

In [ ]:
result_naive = fit(mapper, obs_maps, pot_params_init, disk_df_params_init,
                   bulge_df_params_init, n_steps=300, **FIT_KW)
errs_naive = report("NO annealing (fixed bandwidth)", result_naive, truth)

## 6. Stage 1 — Adam, with Annealing + Matched Blurring

Now the coarse-to-fine scheme, done correctly:

- `anneal_bandwidth=(8.0, OBS_BANDWIDTH)` decays the KDE bandwidth from 8 kpc toward the data's own resolution, so early on the maps overlap and there is a gradient everywhere.
- The schedule is a **ramp followed by a hold**, not a plain log-linear decay all the way down (`anneal_hold_frac`, default 0.35). Matching the observation's resolution means blurring it by `sqrt(h**2 - obs_bandwidth**2)`, and `blur_maps` operates on the *binned* grid, so it cannot represent a blur much narrower than a pixel: the kernel collapses toward the identity, the data stops being blurred while the model's KDE genuinely is smoother, and a systematic mismatch appears that does **not** vanish at the true parameters. Measured here (2.14 kpc pixels): the data-fit loss at the *true* parameters sits at a 0.013 floor when the required blur is ~0.9 pixels, climbs to **0.35** at ~0.17 pixels, and is exactly **0** once the blur reaches zero. So the schedule ramps down only to where the blur is still ~0.5 pixels (h ~ 1.2 kpc) and then jumps straight to `OBS_BANDWIDTH`, where the objective is exact, holding there for the rest of the run. Passing slowly through that band instead optimizes a corrupted objective and injects noisy gradients (final loss 0.0132 and 1.5% median potential error, against 0.0075 and 0.5% for the ramp+hold).
- `obs_bandwidth=OBS_BANDWIDTH` blurs the **observation** to the model's current resolution at every step, so each stage compares maps at matched resolution and the objective stays minimized at the truth.

Annealing is not optional here — removing it is exactly the plateau in Section 5.
This is stage 1 of the optimizer. It ends holding at `OBS_BANDWIDTH`, which is where Section 7 picks up.


In [ ]:
result_adam = fit(mapper, obs_maps, pot_params_init, disk_df_params_init,
                  bulge_df_params_init, n_steps=700,
                  anneal_bandwidth=(8.0, OBS_BANDWIDTH),
                  obs_bandwidth=OBS_BANDWIDTH, **FIT_KW)
errs_adam = report("STAGE 1: Adam, annealed + matched blur, physics ON", result_adam, truth)
print(f"\n(for reference, the ground truth's own Poisson residual is {sc['penalty_final']:.4f})")


## 7. Stage 2 — L-BFGS Polish at Fixed Bandwidth

`lbfgs_refine` below wraps `optax.lbfgs` (memory-limited BFGS with a Wolfe **zoom line search**) around the same loss closure the Adam stage used. Three details make it applicable to *this* loss rather than a textbook one:

- **Fixed bandwidth, identical objective.** The loss is rebuilt with `make_loss_fn` from the same kwargs the Adam stage passed to `fit` and evaluated at `soft_bin_h=OBS_BANDWIDTH`. There the required blur of the observation is exactly zero, so the objective is the exact one — and it is the same objective Adam's hold phase was already descending, which is what makes the two stages composable and their losses directly comparable.
- **Frozen parameters are deleted, not zeroed.** `fit` freezes by zeroing gradients, which is enough for Adam. For L-BFGS a zeroed coordinate would still enter the curvature pairs and the line search, so the four unconstrained parameters are removed from the optimized vector entirely and reinserted afterwards.
- **Bounds by projection.** L-BFGS is unconstrained; the box constraints are enforced by the same clip `fit` applies, after each accepted step. The next iteration builds its curvature pair from the projected point (projected L-BFGS). If the projection is ever active the run says so — it means the fit is pressed against a bound and the quasi-Newton guarantees no longer strictly hold.

**Cost.** An L-BFGS iteration is not an Adam step: the line search evaluates the loss and its gradient several times per iteration, so expect roughly 3-8x an Adam step. The stopping rules (gradient below `grad_tol`, or a stalled line search) usually end the run well before `n_steps`.

### Polishing both Adam endpoints

The refinement is applied twice — to the **plateaued** fit of Section 5 as well as to the **annealed** fit of Section 6 — because that is what separates two explanations of the plateau that the Adam-only comparison cannot tell apart:

- if the plateau is an **optimizer** failure (a badly conditioned surface Adam creeps across), a quasi-Newton method with a line search should make real progress on it;
- if it is an **information** failure (the masked log-mass term has no gradient at all when the model and observed maps barely overlap), then there is no curvature to exploit either, and L-BFGS will stall almost immediately.

Section 5 argues the second. The prediction is therefore that `no annealing + L-BFGS` improves far less than `annealed + L-BFGS` and does not reach it — annealing is not something a better optimizer can substitute for. The four fits are all comparable because they share one objective: `fit` without `anneal_bandwidth` passes `soft_bin_h=None`, and `bin_maps` then defaults to `0.25*max(dx, dz)`, which is exactly `OBS_BANDWIDTH` — the same bandwidth the annealed run holds at and the same one L-BFGS uses.


In [ ]:
def _split_frozen(tree, frozen_params):
    """Splits a params_log pytree into its free and frozen leaves."""
    free = {g: {k: v for k, v in grp.items() if k not in frozen_params}
            for g, grp in tree.items()}
    fixed = {g: {k: v for k, v in grp.items() if k in frozen_params}
             for g, grp in tree.items()}
    return free, fixed


def lbfgs_refine(loss_fn, init_params_log, soft_bin_h=None, param_bounds=None,
                 frozen_params=(), n_steps=100, memory_size=10,
                 max_linesearch_steps=30, grad_tol=1e-7, stall_tol=1e-9,
                 max_stalls=3, verbose_every=10):
    """L-BFGS refinement of a `params_log` pytree at a FIXED bandwidth.

    Second-order (quasi-Newton) polish for the point Adam leaves behind. Adam takes a
    fixed-size step along a rescaled gradient and cannot exploit curvature, so on the
    long narrow valleys this loss has it creeps; L-BFGS builds a low-rank inverse-Hessian
    estimate from the last `memory_size` (step, gradient-change) pairs and picks both the
    direction and — via a Wolfe zoom line search — the distance.

    Three details make it applicable to this particular loss:

    - **Fixed bandwidth.** L-BFGS's curvature memory is only meaningful for one fixed
      objective, so this never anneals. Pass the bandwidth the Adam run ended at.
    - **Frozen parameters are removed** from the optimized vector rather than having
      their gradients zeroed. A zeroed gradient still enters the curvature pairs and
      the line search; deleting the coordinate is exact.
    - **Bounds are enforced by projection** after each accepted step (the same clip
      `fit` applies). The next iteration then builds its curvature pair from the
      projected point, i.e. projected L-BFGS. If the projection is ever active this is
      no longer plain L-BFGS, so it is counted and reported.

    Returns the same dict shape as `pipeline.fit` (`pot_params`, `disk_df_params`,
    `bulge_df_params`, `history`) plus `n_iter`, `stop_reason`, `n_projected` and
    `params_log`, so every downstream cell works on it unchanged.

    NOTE on the history: `fit` records the loss *before* each update, so entry i is the
    loss at the parameters iteration i started from. This does the same and then appends
    one final entry evaluated at the returned parameters, so `history['loss'][-1]` is the
    loss of the parameters actually handed back (an L-BFGS line search can take a large
    final step, and reporting the pre-step value would understate it).
    """
    free, fixed = _split_frozen(init_params_log, frozen_params)
    log_lo, log_hi = log_bounds_tree(init_params_log, param_bounds)
    lo_free, _ = _split_frozen(log_lo, frozen_params)
    hi_free, _ = _split_frozen(log_hi, frozen_params)

    h = None if soft_bin_h is None else jnp.asarray(soft_bin_h, jnp.float32)

    def merge(free_tree):
        return {g: {**fixed[g], **free_tree[g]} for g in init_params_log}

    def loss_aux(free_tree):
        return loss_fn(merge(free_tree), h)

    def value_fn(free_tree):          # scalar-only: this is what the line search calls
        return loss_aux(free_tree)[0]

    opt = optax.lbfgs(
        memory_size=memory_size,
        linesearch=optax.scale_by_zoom_linesearch(
            max_linesearch_steps=max_linesearch_steps),
    )

    @jax.jit
    def step(free_tree, state):
        (value, aux), grad = jax.value_and_grad(loss_aux, has_aux=True)(free_tree)
        # Same guard as `fit`: a single non-finite entry would otherwise poison the
        # curvature memory and every later direction.
        grad = jax.tree_util.tree_map(lambda g: jnp.where(jnp.isfinite(g), g, 0.0), grad)
        updates, state = opt.update(grad, state, free_tree,
                                    value=value, grad=grad, value_fn=value_fn)
        stepped = optax.apply_updates(free_tree, updates)
        clipped = jax.tree_util.tree_map(jnp.clip, stepped, lo_free, hi_free)
        projected = ravel_pytree(jax.tree_util.tree_map(
            lambda a, b: jnp.abs(a - b), stepped, clipped))[0].max()
        gnorm = jnp.abs(ravel_pytree(grad)[0]).max()
        return clipped, state, value, aux, gnorm, projected

    @jax.jit
    def evaluate(free_tree):
        return loss_aux(free_tree)

    history = {k: [] for k in ('loss', 'mass_loss', 'vrot_loss', 'sigma_loss',
                               'poisson_penalty', 'reg', 'pot_params',
                               'disk_df_params', 'bulge_df_params')}

    def record(value, aux, free_tree):
        pot, disk, bulge = log_to_params(merge(free_tree))
        history['loss'].append(float(value))
        for k in ('mass_loss', 'vrot_loss', 'sigma_loss', 'poisson_penalty', 'reg'):
            history[k].append(float(aux[k]))
        history['pot_params'].append({k: float(v) for k, v in pot.items()})
        history['disk_df_params'].append({k: float(v) for k, v in disk.items()})
        history['bulge_df_params'].append({k: float(v) for k, v in bulge.items()})

    state = opt.init(free)
    best_free, best_loss = free, np.inf
    stalls, n_projected, stop_reason = 0, 0, f'reached n_steps={n_steps}'
    prev = np.inf

    for i in range(n_steps):
        free, state, value, aux, gnorm, projected = step(free, state)
        value, gnorm = float(value), float(gnorm)
        record(value, aux, free)      # loss BEFORE this step, params AFTER it
        if projected > 0:
            n_projected += 1
        if not np.isfinite(value):
            stop_reason = f'non-finite loss at iteration {i}'
            break
        if value < best_loss:
            best_loss, best_free = value, free
        if verbose_every and (i % verbose_every == 0 or i == n_steps - 1):
            print(f"  lbfgs {i:4d}   loss {value:.6f}   |grad|_inf {gnorm:.3e}")
        if gnorm < grad_tol:
            stop_reason = f'gradient below {grad_tol:g} at iteration {i}'
            break
        # A failed line search returns a ~zero step, so the loss stops moving.
        if abs(prev - value) < stall_tol * max(1.0, abs(prev)):
            stalls += 1
            if stalls >= max_stalls:
                stop_reason = f'no progress for {max_stalls} iterations (line search stalled)'
                break
        else:
            stalls = 0
        prev = value

    # Final entry, at the parameters actually returned (see the docstring).
    final_value, final_aux = evaluate(best_free)
    record(final_value, final_aux, best_free)

    pot, disk, bulge = log_to_params(merge(best_free))
    if verbose_every:
        print(f"  stopped: {stop_reason}")
        if n_projected:
            print(f"  WARNING: bounds projection active on {n_projected} iterations "
                  f"-- the refinement is projected L-BFGS, not plain L-BFGS")
    return {
        'pot_params': pot, 'disk_df_params': disk, 'bulge_df_params': bulge,
        'params_log': merge(best_free), 'history': history,
        'n_iter': len(history['loss']) - 1, 'stop_reason': stop_reason,
        'n_projected': n_projected,
    }


In [ ]:
# The objective must be *identical* to the one Adam was descending, so it is rebuilt
# from FIT_KW rather than re-specified (FIT_KW's two optimizer-only keys are dropped).
LOSS_KW = {k: v for k, v in FIT_KW.items() if k not in ('learning_rate', 'frozen_params')}
loss_fn_final = make_loss_fn(mapper, obs_maps, obs_bandwidth=OBS_BANDWIDTH, **LOSS_KW)

N_LBFGS_STEPS = 150
LBFGS_MEMORY = 10


def objective_at(res):
    """The final fixed-bandwidth objective at a fit's parameters. `fit` records the loss
    *before* each update, so its last history entry is one step stale; this evaluates the
    parameters actually returned, which is what makes the four fits comparable."""
    p_log = params_to_log(res['pot_params'], res['disk_df_params'], res['bulge_df_params'])
    return float(loss_fn_final(p_log, OBS_BANDWIDTH)[0])


def polish(tag, res):
    p_log = params_to_log(res['pot_params'], res['disk_df_params'], res['bulge_df_params'])
    print(f"--- {tag}")
    out = lbfgs_refine(loss_fn_final, p_log, soft_bin_h=OBS_BANDWIDTH,
                       frozen_params=FROZEN, n_steps=N_LBFGS_STEPS,
                       memory_size=LBFGS_MEMORY, verbose_every=10)
    return out


# Both Adam endpoints get the same polish, on the same objective.
result_naive_lbfgs = polish('no annealing   ->  L-BFGS', result_naive)
result_lbfgs = polish('annealed Adam  ->  L-BFGS', result_adam)

errs_naive_lbfgs = report("\nNO annealing + L-BFGS", result_naive_lbfgs, truth)
errs_lbfgs = report("\nSTAGE 2: annealed Adam + L-BFGS", result_lbfgs, truth)

print("\nfinal objective, all four fits evaluated at the SAME fixed bandwidth:")
rows = [('no annealing', result_naive), ('no annealing + L-BFGS', result_naive_lbfgs),
        ('annealed Adam', result_adam), ('annealed Adam + L-BFGS', result_lbfgs)]
for tag, r in rows:
    print(f"  {tag:24s} {objective_at(r):.6f}")
print(f"\nL-BFGS iterations:  from the plateau {result_naive_lbfgs['n_iter']}"
      f" ({result_naive_lbfgs['stop_reason']})"
      f"   |   from the annealed fit {result_lbfgs['n_iter']}"
      f" ({result_lbfgs['stop_reason']})")


## 8. Convergence

In [ ]:
h_nv, h_nl = result_naive['history'], result_naive_lbfgs['history']
h_ad, h_lb = result_adam['history'], result_lbfgs['history']
x_nl = np.arange(len(h_nv['loss']), len(h_nv['loss']) + len(h_nl['loss']))
x_lb = np.arange(len(h_ad['loss']), len(h_ad['loss']) + len(h_lb['loss']))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(h_nv['loss'], color='tab:red', label='no annealing (Adam)')
axes[0].plot(x_nl, h_nl['loss'], color='tab:orange', label='no annealing + L-BFGS')
axes[0].plot(h_ad['loss'], color='tab:green', label='annealed Adam')
axes[0].plot(x_lb, h_lb['loss'], color='tab:blue', label='annealed Adam + L-BFGS')
for x, c in [(x_nl[0], 'tab:orange'), (x_lb[0], 'tab:blue')]:
    axes[0].axvline(x, color=c, ls='--', lw=1, alpha=0.5)
axes[0].set_yscale('log'); axes[0].set_xlabel('iteration'); axes[0].set_ylabel('total loss')
axes[0].set_title('Dashed lines mark each L-BFGS handoff'); axes[0].legend(fontsize=8)

for key, lab in [('mass_loss', 'mass'), ('vrot_loss', 'v_rot'),
                 ('sigma_loss', 'sigma'), ('poisson_penalty', 'poisson')]:
    line, = axes[1].plot(h_ad[key], label=lab, alpha=0.85)
    axes[1].plot(x_lb, h_lb[key], color=line.get_color(), alpha=0.85)
axes[1].axvline(x_lb[0], color='gray', ls='--', lw=1)
axes[1].axhline(sc['penalty_final'], color='gray', ls=':', lw=1,
                label="truth's own poisson residual")
axes[1].set_yscale('log'); axes[1].set_xlabel('iteration'); axes[1].set_ylabel('loss component')
axes[1].set_title('Components, annealed chain'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

# Both L-BFGS segments are at fixed bandwidth, so unlike the annealed Adam phase their
# losses ARE comparable across iterations: these are real improvements in fit, not a
# change in what is being measured.
for tag, hh, res in [('from the plateau     ', h_nl, result_naive_lbfgs),
                     ('from the annealed fit', h_lb, result_lbfgs)]:
    print(f"L-BFGS {tag}: {hh['loss'][0]:.6f} -> {hh['loss'][-1]:.6f}"
          f"   (factor {hh['loss'][0] / max(hh['loss'][-1], 1e-30):.2f}"
          f", {res['n_iter']} iterations)")
    if res['n_projected']:
        print(f"    bounds projection active on {res['n_projected']} iterations")


Three features of the loss trace are worth understanding.

First, **an annealed loss is not comparable across iterations**: as the bandwidth shrinks the comparison gets stricter, so the same-quality fit yields a larger loss. A rising annealed loss does not by itself mean the fit is degrading.

Second, this run **should not** show the large mid-run rise and violent oscillation that a plain log-linear schedule produces (loss climbing from ~0.012 to ~0.5 between steps 400 and 680). That was not physics: it was the sub-pixel blurring artifact described in Section 6, confirmed by the fact that quadrupling the tracer count left it unchanged. The ramp-and-hold schedule skips the affected bandwidth band entirely, so the curve should descend and then flatten on the exact objective.

Third, the two stages are **not** read the same way. Everything left of the handoff line is annealed, so it is subject to the caveat above. Everything right of it is at one fixed bandwidth on the exact objective, so the L-BFGS segment is a like-for-like comparison across its own iterations and its descent is a genuine improvement in fit. The step down at the handoff itself is not a discontinuity in the objective — both sides evaluate the same one — it is L-BFGS's first line search taking a step Adam's fixed learning rate could not.

The Poisson curve is the one to watch: it settles at roughly the ground truth's *own* residual (dotted line) instead of being driven far below it. That is the signature of a self-consistency term that is compatible with the data rather than competing with it.

## 9. Map Comparison

In [ ]:
adam_maps = generate_edge_on_maps(mapper, result_adam['pot_params'],
                                  result_adam['disk_df_params'],
                                  result_adam['bulge_df_params'], **GRID_KW)
final_maps = generate_edge_on_maps(mapper, result_lbfgs['pot_params'],
                                   result_lbfgs['disk_df_params'],
                                   result_lbfgs['bulge_df_params'], **GRID_KW)
naive_maps = generate_edge_on_maps(mapper, result_naive['pot_params'],
                                   result_naive['disk_df_params'],
                                   result_naive['bulge_df_params'], **GRID_KW)
naive_lbfgs_maps = generate_edge_on_maps(mapper, result_naive_lbfgs['pot_params'],
                                         result_naive_lbfgs['disk_df_params'],
                                         result_naive_lbfgs['bulge_df_params'], **GRID_KW)


def residual_maps(model, obs, mask):
    """Model minus observation, in the units each term is actually compared in by the
    loss: mass as a log10 ratio (dex), the kinematics as plain differences (km/s)."""
    mm, om = np.array(model['mass']), np.array(obs['mass'])
    floor = 1e-4 * np.nanmax(om)          # keeps log10 finite in the faintest pixels
    d_mass = np.log10(np.maximum(mm, floor)) - np.log10(np.maximum(om, floor))
    d_vrot = np.array(model['v_rot']) - np.array(obs['v_rot'])
    d_sigma = np.array(model['sigma']) - np.array(obs['sigma'])
    return [np.where(mask, d, np.nan) for d in (d_mass, d_vrot, d_sigma)]


def plot_residual_row(axes, model, obs, mask, title_prefix='Fit - Obs'):
    """Diverging panels, symmetric about zero. The colour limits come from the data
    (98th percentile) so structure stays visible even for a good fit, and the RMS is
    printed in each title so a saturated-looking panel cannot be misread as a big error."""
    extent = [-EXTENT_X, EXTENT_X, -EXTENT_Z, EXTENT_Z]
    resids = residual_maps(model, obs, mask)
    labels = [('mass', 'dex', 0.05), ('v_rot', 'km/s', 5.0), ('sigma', 'km/s', 5.0)]
    ims = []
    for ax, d, (name, unit, floor) in zip(axes, resids, labels):
        finite = d[np.isfinite(d)]
        lim = max(float(np.percentile(np.abs(finite), 98)) if finite.size else floor, floor)
        rms = float(np.sqrt(np.nanmean(d ** 2)))
        im = ax.imshow(d, origin='lower', extent=extent, cmap='RdBu_r',
                       vmin=-lim, vmax=lim, aspect='auto')
        ax.set_title(f'{title_prefix}: {name}\n(RMS {rms:.3g} {unit})', fontsize=10)
        ims.append(im)
    return ims


fig, axes = plt.subplots(3, 7, figsize=(31, 10))
for col, (name, m) in enumerate([('Observed', obs_maps), ('Far init', init_maps),
                                 ('No annealing', naive_maps),
                                 ('No anneal +L-BFGS', naive_lbfgs_maps),
                                 ('Annealed Adam', adam_maps),
                                 ('Annealed +L-BFGS', final_maps)]):
    plot_maps_row(axes[:, col], m, mass_mask, name, vmax_mass)

# Last column: final (annealed Adam + L-BFGS) fit minus observation. It gets its own
# colourbars because the residual scale is completely different from the maps'.
res_ims = plot_residual_row(axes[:, 6], final_maps, obs_maps, mass_mask,
                            title_prefix='Annealed+L-BFGS - Obs')
for im, ax, lab in zip(res_ims, axes[:, 6], ['dex', 'km/s', 'km/s']):
    fig.colorbar(im, ax=ax, label=lab)

plt.tight_layout(); plt.show()

# The same information as numbers, over the detected pixels, for both stages -- this
# is where any gain from the L-BFGS polish shows up in observable units.
for tag, mp in [('No annealing      ', naive_maps),
                ('No anneal +L-BFGS ', naive_lbfgs_maps),
                ('Annealed Adam     ', adam_maps),
                ('Annealed +L-BFGS  ', final_maps)]:
    d_mass, d_vrot, d_sigma = residual_maps(mp, obs_maps, mass_mask)
    print(f"{tag} minus observation, over the detected pixels:")
    for nm, d, unit in [('mass ', d_mass, 'dex'), ('v_rot', d_vrot, 'km/s'),
                        ('sigma', d_sigma, 'km/s')]:
        print(f"  {nm}: RMS {np.sqrt(np.nanmean(d**2)):8.4f} {unit:4s}  "
              f"median {np.nanmedian(d):+8.4f}  max|.| {np.nanmax(np.abs(d)):8.4f}")

## 10. Parameter Recovery

The potential parameters and the DF parameters are shown separately, because they are constrained very differently: at the truth, the gradients with respect to the potential parameters are ~1000x larger than those with respect to the DF *shape* parameters. The four frozen parameters are listed for completeness but are structurally unrecoverable from these maps.

In [ ]:
all_init = {**pot_params_init, **disk_df_params_init, **bulge_df_params_init}
fitted = {**result_lbfgs['pot_params'], **result_lbfgs['disk_df_params'],
          **result_lbfgs['bulge_df_params']}

groups = [('potential', list(pot_params_true)),
          ('disk DF', list(disk_df_params_true)),
          ('bulge DF', list(bulge_df_params_true))]

print(f"{'parameter':16s} {'true':>12s} {'final':>12s} {'init err':>9s} "
      f"{'naive':>7s} {'nv+lbf':>7s} {'adam':>7s} {'ad+lbf':>7s}")
for gname, keys in groups:
    print(f"-- {gname}")
    for k in keys:
        ie = abs(all_init[k] - truth[k]) / abs(truth[k]) * 100
        flag = '  (frozen)' if k in FROZEN else ''
        print(f"{k:16s} {truth[k]:12.4g} {float(fitted[k]):12.4g} "
              f"{ie:8.0f}% {errs_naive[k]:6.0f}% {errs_naive_lbfgs[k]:6.0f}% "
              f"{errs_adam[k]:6.0f}% {errs_lbfgs[k]:6.0f}%{flag}")

keys = [k for k in truth if k not in FROZEN]
yv = np.arange(len(keys))
colors = ['tab:blue' if k in pot_params_true else
          'tab:orange' if k in disk_df_params_true else 'tab:green' for k in keys]
series = [('far init', [abs(all_init[k] - truth[k]) / abs(truth[k]) * 100 for k in keys], 'lightgray'),
          ('no annealing', [errs_naive[k] for k in keys], 'tab:red'),
          ('no annealing + L-BFGS', [errs_naive_lbfgs[k] for k in keys], 'salmon'),
          ('annealed Adam', [errs_adam[k] for k in keys], 'darkgray'),
          ('annealed + L-BFGS (blue=pot, orange=disk, green=bulge)',
           [errs_lbfgs[k] for k in keys], colors)]
hgt = 0.16
fig, ax = plt.subplots(figsize=(10, 9))
for j, (lab, vals, col) in enumerate(series):
    ax.barh(yv + (j - 2) * hgt, vals, height=hgt, color=col, label=lab)
ax.set_yticks(yv); ax.set_yticklabels(keys); ax.invert_yaxis()
ax.set_xscale('log'); ax.axvline(20, color='gray', ls=':', lw=1)
ax.set_xlabel('relative error vs. ground truth [%]')
ax.set_title('Parameter recovery: does an L-BFGS polish substitute for annealing?')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()

pot_keys = list(pot_params_true)
print(f"{'':26s} {'all constrained':>16s} {'potential only':>16s}")
for tag, e in [('no annealing', errs_naive), ('no annealing + L-BFGS', errs_naive_lbfgs),
               ('annealed Adam', errs_adam), ('annealed Adam + L-BFGS', errs_lbfgs)]:
    print(f"median error  {tag:24s} {np.median([e[k] for k in keys]):9.1f}%"
          f" {np.median([e[k] for k in pot_keys]):15.1f}%")


## 11. Summary

What this notebook changes relative to `optimization_pipeline.ipynb`, and why each mattered:

| change | reason |
|---|---|
| the observation is generated from a **self-consistent** model | picking the potential and DF independently leaves a real Poisson residual at the "truth", so the truth is not a minimum and the physics term pulls the fit away from it |
| the penalty compares against the **baryonic** potential | the tracers carry `M_disk + M_bulge`; they cannot reproduce a dark halo that outweighs them ~17x |
| the penalty is **masked** to pixels with tracer support | empty pixels gave `log10(0 + 1e-10) = -10`, i.e. multi-dex artefacts over two thirds of the grid, which dominated the value |
| `obs_bandwidth=` blurs the **data** during annealing | otherwise only the model is smoothed and the wide-bandwidth stages optimize a biased objective |
| `frozen_params=` | `Sigma0`/`N0_spheroid` cancel exactly in the normalized weights; `L0`/`Rinit_for_Rc` do not enter these maps. Zero gradient means they can only drift |
| annealing is kept | without it the fit plateaus and the potential is not recovered at all (Section 5) |
| the optimizer is **two-stage**: Adam then L-BFGS | Adam is robust to the noisy, badly scaled gradients far from the solution and is the only option while the objective is still changing under annealing; it is also the wrong tool at the end, where a fixed-size step along a curvature-blind direction crawls along a narrow valley. L-BFGS uses a curvature estimate and a line search, so it converges where Adam creeps — but only on a fixed objective, which is why it runs after the anneal, not during it |

**What is genuinely recovered:** the potential parameters, to a few percent, from a start that was 430% off.

**What is not:** the DF *shape* parameters (`Rd`, `RsigR`, `RsigZ`, the spheroid slopes). This is not an optimizer failure — a single edge-on projection genuinely under-constrains them, and their gradients are orders of magnitude smaller than the potential's. Breaking that degeneracy needs more information: additional viewing angles, higher-order (Gauss-Hermite `h3`/`h4`) kinematics, or informative priors on the DF shape.

**Using real observations:** build `obs_maps` with the same keys (`'mass'`, `'v_rot'`, `'sigma'`, `'x_edges'`, `'z_edges'`) on a grid matching `grid_size`/`extent_x`/`extent_z`, set `obs_bandwidth` to the effective resolution of those maps (or the PSF width you smoothed them to), and call `fit(...)`. Note that Sections 1-2 are a *mock-building* step: with real data there is no ground truth to make self-consistent, so the self-consistency penalty is then a genuine physical prior on the model rather than something you can pre-satisfy.
**On the second stage specifically.** The gain to check is the one in Section 8: the fixed-bandwidth objective before and after the polish, which is a like-for-like number. The `no annealing + L-BFGS` run is the control — the two chains start from the same initial guess and end on the same objective, so the gap between them at the end is the part of the result that annealing, and not the optimizer, is responsible for. Two things limit how far the polish can go:

- **Float32.** Everything runs in single precision (`params_to_log` casts explicitly), so gradients are accurate to ~1e-7 relative. A quasi-Newton method converges quadratically only until it hits that floor, and the line search can fail early once the gradient noise is comparable to the true gradient. That is what the `line search stalled` stop reason means when it appears — it is a precision limit, not a bug. Enabling `jax_enable_x64` would push it further, at roughly 2x the memory and a large slowdown on GPU.
- **The degeneracies do not go away.** L-BFGS finds the minimum of *this* objective more precisely; it does not add information. The DF shape parameters are flat directions, and a better-converged fit along a flat direction is still an arbitrary point on it. Expect the potential parameters to improve and the shape parameters not to.
